### Final check of options through configs (A) ~ (D)

In [12]:
import os
from pathlib import Path

import numpy as np

from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names",
    category=UserWarning,
)

In [13]:
EMB = Path("./data_tox/Embeddings")

def load_emb(name: str) -> np.ndarray:
    return np.load(EMB / name)

def rmse(y_true, y_pred) -> float:
    yt = np.asarray(y_true, dtype=np.float64).ravel()
    yp = np.asarray(y_pred, dtype=np.float64).ravel()
    return float(np.sqrt(mean_squared_error(yt, yp)))

ecfp_blind = load_emb("blind_morgan.npy")
mlm_blind = load_emb("blind_mlm.npy")
y_blind = load_emb("blind_labels.npy").astype(np.float64).reshape(-1)
ecfp_ld = load_emb("leaderboard_morgan.npy")
mlm_ld = load_emb("leaderboard_mlm.npy")
y_ld = load_emb("leaderboard_labels.npy").astype(np.float64).reshape(-1)

In [14]:
train_ecfp = np.load(EMB / "train_morgan.npy")
train_mlm = np.load(EMB / "train_mlm.npy")
train_labels = np.load(EMB / "train_labels.npy")

train_mlm_aug = np.load(
    EMB / "train_mlm_aug_trial_200_sampled_n110.npy")
train_mlm_aug_labels = np.load(
    EMB / "train_mlm_labels_aug_trial_200_sampled_n110.npy").reshape(-1)

scaler = StandardScaler()
train_mlms = scaler.fit_transform(train_mlm)
mlm_blinds = scaler.transform(mlm_blind)
mlm_lds = scaler.transform(mlm_ld)

scaler = StandardScaler()
train_mlm_augs = scaler.fit_transform(train_mlm_aug)
mlm_blind_augs = scaler.transform(mlm_blind)
mlm_ld_augs = scaler.transform(mlm_ld)

### (A) ECFP4 + RF

In [16]:
rf_default = RandomForestRegressor(random_state=42, n_jobs=10)
rf_default.fit(train_ecfp, train_labels)

print(f"Blind RMSE: {rmse(y_blind, rf_default.predict(ecfp_blind))}")
print(f"Leaderboard RMSE: {rmse(y_ld, rf_default.predict(ecfp_ld))}")

Blind RMSE: 25.02494152808374
Leaderboard RMSE: 9.846861571557538


### (B) MLM + LGBM

In [17]:
lgbm_default = LGBMRegressor(
    random_state=42,
    n_jobs=10,
    verbose=-1,
)

lgbm_default.fit(train_mlms, train_labels)
print(f"Blind RMSE: {rmse(y_blind, lgbm_default.predict(mlm_blinds))}")
print(f"Leaderboard RMSE: {rmse(y_ld, lgbm_default.predict(mlm_lds))}")

Blind RMSE: 24.90970374534041
Leaderboard RMSE: 2.691012823009115


### (C) MLM + LGBM + Augmentation

In [18]:
lgbm_default = LGBMRegressor(
    random_state=42,
    n_jobs=10,
    verbose=-1,
)

lgbm_default.fit(train_mlm_augs, train_mlm_aug_labels)

print(f"Blind RMSE: {rmse(y_blind, lgbm_default.predict(mlm_blind_augs))}")
print(f"Leaderboard RMSE: {rmse(y_ld, lgbm_default.predict(mlm_ld_augs))}")

Blind RMSE: 23.099077267445097
Leaderboard RMSE: 2.7359566449990593


### (D) MLM + LGBM (tuned) + Augmentation

In [21]:
par_s = {
    "colsample_bytree": 1.0,
    "learning_rate": 0.05,
    "max_depth": -1,
    "min_child_samples": 20,
    "num_leaves": 31,
    "reg_alpha": 0.0,
    "reg_lambda": 0.1,
    "subsample": 1.0,
}
lgbm_tuned = LGBMRegressor(
    random_state=42,
    n_jobs=10,
    verbose=-1,
    n_estimators=158,
    **par_s,
)

lgbm_tuned.fit(train_mlm_augs, train_mlm_aug_labels)

mlm_pred = lgbm_tuned.predict(mlm_blind_augs)
br_s = rmse(y_blind, mlm_pred)
ld_s = rmse(y_ld, lgbm_tuned.predict(mlm_ld_augs))

print(f"Blind RMSE: {rmse(y_blind, lgbm_tuned.predict(mlm_blind_augs))}")
print(f"Leaderboard RMSE: {rmse(y_ld, lgbm_tuned.predict(mlm_ld_augs))}")

Blind RMSE: 22.734914315306728
Leaderboard RMSE: 3.8625891844605045
